# Lesson 07 · Limits & Colimits

Terminal objects, products, and their cousins were the same construction wearing different diagrams. Time to say so — and prove the uniqueness theorem exactly once.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-07-limits-and-colimits.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Limits by hand

**`setup.py`** — carried over from Lessons 1, 2 and 6

In [1]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

class Functor:
    """F : C -> D, given by a map on objects and a map on arrows."""
    def __init__(self, C, D, on_obj, on_arr):
        self.C, self.D, self.on_obj, self.on_arr = C, D, on_obj, on_arr

    def check(self):
        C, D, Fo, Fa = self.C, self.D, self.on_obj, self.on_arr
        for f in C.ar:                        # typing: F f : F X -> F Y
            X, Y = C.ar[f]
            assert D.ar[Fa[f]] == (Fo[X], Fo[Y])
        for X in C.obj:                       # preserves identities
            assert Fa[C.id[X]] == D.id[Fo[X]]
        for g, f in product(C.ar, repeat=2):  # preserves composition
            if C.composable(g, f):
                assert Fa[C.comp[(g, f)]] == D.comp[(Fa[g], Fa[f])]
        return True

def arrows_between(C, X, Y):
    "all arrows X -> Y of the finite category C"
    return [a for a, (d, c) in C.ar.items() if d == X and c == Y]

from itertools import product as cartesian

# the divisibility order on {1, 2, 3, 6}, as in Lesson 6
objs = {1, 2, 3, 6}
ar   = {f"{a}|{b}": (a, b) for a in objs for b in objs if b % a == 0}
comp = {(g, f): f"{ar[f][0]}|{ar[g][1]}"
        for g in ar for f in ar if ar[f][1] == ar[g][0]}
Div  = FiniteCategory(objs, ar, comp, {n: f"{n}|{n}" for n in objs})

**`limits.py`** — the general limit finder

In [2]:
def cones(C, D):
    "All cones over the diagram D : J -> C, as (apex, legs) pairs."
    J, out = D.C, []
    Jobjs = sorted(J.obj, key=str)
    for X in C.obj:
        legsets = [arrows_between(C, X, D.on_obj[j]) for j in Jobjs]
        for combo in cartesian(*legsets):
            legs = dict(zip(Jobjs, combo))
            # every triangle commutes:  D(a) . leg_i == leg_j
            if all(C.comp[(D.on_arr[a], legs[i])] == legs[j]
                   for a, (i, j) in J.ar.items()):
                out.append((X, legs))
    return out

def find_limits(C, D):
    "Limit = the cone through which every cone factors exactly once."
    cs = cones(C, D)
    return [(apex, legs) for apex, legs in cs
            if all(len([m for m in arrows_between(C, X, apex)
                        if all(C.comp[(legs[j], m)] == l[j]
                               for j in legs)]) == 1
                   for X, l in cs)]

**`colimits.py`** — the arrow-reversed twin

In [3]:
def cocones(C, D):
    "All cocones under D, as (apex, legs) pairs: legs run D(j) -> apex."
    J, out = D.C, []
    Jobjs = sorted(J.obj, key=str)
    for X in C.obj:
        legsets = [arrows_between(C, D.on_obj[j], X) for j in Jobjs]
        for combo in cartesian(*legsets):
            legs = dict(zip(Jobjs, combo))
            # every triangle commutes:  leg_j . D(a) == leg_i
            if all(C.comp[(legs[j], D.on_arr[a])] == legs[i]
                   for a, (i, j) in J.ar.items()):
                out.append((X, legs))
    return out

def find_colimits(C, D):
    "Colimit = the cocone that factors through every cocone exactly once."
    cs = cocones(C, D)
    return [(apex, legs) for apex, legs in cs
            if all(len([m for m in arrows_between(C, apex, X)
                        if all(C.comp[(m, legs[j])] == l[j]
                               for j in legs)]) == 1
                   for X, l in cs)]

**`zoo.py`** — three shapes, one function

In [4]:
Empty = FiniteCategory(set(), {}, {}, {})

Disc2 = FiniteCategory({0, 1}, {'i0':(0,0), 'i1':(1,1)},
                       {('i0','i0'):'i0', ('i1','i1'):'i1'},
                       {0:'i0', 1:'i1'})

cos_ar   = {'i0':(0,0),'i1':(1,1),'i2':(2,2),'u':(0,2),'v':(1,2)}
cos_comp = {('i0','i0'):'i0',('i1','i1'):'i1',('i2','i2'):'i2',
            ('u','i0'):'u',('i2','u'):'u',('v','i1'):'v',('i2','v'):'v'}
Cospan = FiniteCategory({0,1,2}, cos_ar, cos_comp, {0:'i0',1:'i1',2:'i2'})

D_empty  = Functor(Empty,  Div, {}, {})
D_23     = Functor(Disc2,  Div, {0:2, 1:3}, {'i0':'2|2', 'i1':'3|3'})
D_cospan = Functor(Cospan, Div, {0:2, 1:3, 2:6},
                   {'i0':'2|2','i1':'3|3','i2':'6|6',
                    'u':'2|6','v':'3|6'})

print("limit of empty diagram :", find_limits(Div, D_empty))
print("limit of {2, 3}        :", find_limits(Div, D_23))
print("limit of 2 -> 6 <- 3   :", find_limits(Div, D_cospan))
print("colimit of {2, 3}      :", find_colimits(Div, D_23))

limit of empty diagram : [(6, {})]
limit of {2, 3}        : [(1, {0: '1|2', 1: '1|3'})]
limit of 2 -> 6 <- 3   : [(1, {0: '1|2', 1: '1|3', 2: '1|6'})]
colimit of {2, 3}      : [(6, {0: '2|6', 1: '3|6'})]


**`set_limits.py`** — equalizers solve, pullbacks join

In [5]:
# equalizer of f, g : X -> Y  ==  the solution set of  f(x) = g(x)
X = range(-3, 4)
f = lambda x: x * x
g = lambda x: x + 2
E = [x for x in X if f(x) == g(x)]
print("equalizer of x\u00b2 and x+2 on [-3..3]:", E)

# pullback of  dept : Emp -> Key  and  key : Bldg -> Key  ==  a join
employees = [("Alice", "eng"), ("Bob", "sales"), ("Carol", "eng")]
buildings = [("eng", "Building 1"), ("sales", "Building 2")]
dept = lambda e: e[1]
key  = lambda b: b[0]

P  = [(e, b) for e in employees for b in buildings if dept(e) == key(b)]
p1 = lambda x: x[0]
p2 = lambda x: x[1]

for e, b in P:
    print(f"{e[0]} works in {b[1]}")
print("square commutes:", all(dept(p1(x)) == key(p2(x)) for x in P))

equalizer of x² and x+2 on [-3..3]: [-1, 2]
Alice works in Building 1
Bob works in Building 2
Carol works in Building 1
square commutes: True


## Exercise 15

**`span.py`**

In [6]:
sp_ar   = {'i0':(0,0),'i1':(1,1),'i2':(2,2),'u':(1,0),'v':(1,2)}
sp_comp = {('i0','i0'):'i0',('i1','i1'):'i1',('i2','i2'):'i2',
            ('u','i1'):'u',('i0','u'):'u',('v','i1'):'v',('i2','v'):'v'}
Span = FiniteCategory({0,1,2}, sp_ar, sp_comp, {0:'i0',1:'i1',2:'i2'})

D_span = Functor(Span, Div, {0:2, 1:1, 2:3},
                 {'i0':'2|2','i1':'1|1','i2':'3|3',
                  'u':'1|2','v':'1|3'})

print(find_limits(Div, D_span))
print(find_colimits(Div, D_span))

[(1, {0: '1|2', 1: '1|1', 2: '1|3'})]
[(6, {0: '2|6', 1: '1|6', 2: '3|6'})]


## Exercise 16

**`fork.py`**

In [7]:
fk_ar = {'iE':('E','E'),'iA':('A','A'),'iB':('B','B'),
         'e':('E','A'),'f':('A','B'),'g':('A','B'),'w':('E','B')}
ident = {'E':'iE','A':'iA','B':'iB'}
fk_comp = {}
for x, (u, v) in fk_ar.items():
    fk_comp[(ident[v], x)] = x
    fk_comp[(x, ident[u])] = x
fk_comp[('f','e')] = 'w'
fk_comp[('g','e')] = 'w'          # both composites land on w
Fork = FiniteCategory({'E','A','B'}, fk_ar, fk_comp, ident)

pp_ar = {'i0':(0,0),'i1':(1,1),'u':(0,1),'v':(0,1)}   # parallel pair
pp_comp = {('i0','i0'):'i0',('i1','i1'):'i1',
           ('u','i0'):'u',('i1','u'):'u',
           ('v','i0'):'v',('i1','v'):'v'}
Pair = FiniteCategory({0,1}, pp_ar, pp_comp, {0:'i0',1:'i1'})

D_fg = Functor(Pair, Fork, {0:'A',1:'B'},
               {'i0':'iA','i1':'iB','u':'f','v':'g'})
print(find_limits(Fork, D_fg))

[('E', {0: 'e', 1: 'w'})]
